In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 벡터 DB : Chroma vs Pinecone
- Chroma : 인메모리 vector DB, 로컬메모리 vector DB
- Pinecone : 클라우드 vector DB
    (Pinecone console에 api key 생성 -> .env (PINECONE_API_KEY등록)

# 0. 패키지 설치

In [ ]:
%pip install -q pinecone-client langchain-pinecone

# 1. Knowledge Base 구성을 위한 데이터 생성

In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)
document_list = loader.load_and_split(text_splitter)

In [ ]:
len(document_list)

In [2]:
# embedding : openAI API text-embedding-3-large
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(
    #model="text-embedding-3-large"
    model="embedding-query"
)

In [16]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
# 데이터를 처음 저장할 때
index_name = "tax-index-upstage"
database = PineconeVectorStore.from_documents(
    documents = document_list,
    embedding = embedding,
    index_name = index_name
)
# 업로드한 벡터 DB 가져올 때
# database = PineconeVectorStore(
#     embedding=embedding, #질문을 임베딩하여 유사도 검색
#     index_name = index_name
# )

CPU times: total: 12.7 s
Wall time: 1min 13s


# 2. 답변 생성을 위한 Retrieval

In [14]:
retriever = database.as_retriever(
    #search_kwargs={'k':4}
)

# 3. 제공되는 prompt를 활용하여 답변 생성

In [17]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [18]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt":prompt}
)

In [19]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
ai_message = qa_chain.invoke({'query':query})
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 공제액이 각각 적용되어 계산됩니다. 근로소득공제는 총급여액에 따라 최대 2000만원까지 공제되고, 자녀세액공제 등 공제 항목도 고려해야 합니다. 정확한 소득세액을 계산하려면 모든 공제 항목을 반영하여 세율에 적용해야 하며, 일반적으로는 대략 7-10% 수준입니다.'}